# Lab 4: Tokenization and Text Processing - The Language Interface

## Lab Overview

This lab explores tokenization, the crucial process that converts human text into numerical representations that neural networks can understand. You'll learn about different tokenization strategies, encoding/decoding processes, and their impact on model performance.

## Learning Objectives

By the end of this lab, you will:
- Understand different tokenization approaches (word, subword, character)
- Master encoding and decoding processes for text
- Learn about vocabulary management and token IDs
- Explore multilingual tokenization challenges
- Set up AMD GPU backend for efficient text processing
- Connect tokenization to transformer model inputs

---

In [2]:
# AMD GPU initialization
try:
    from jupyter_gpu_init import *
    print("AMD GPU environment initialized successfully")
except ImportError:
    print("AMD GPU initialization not found, using CPU fallback")

# Essential Imports and GPU Setup
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, pipeline, AutoModel
import numpy as np

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

AMD GPU environment initialized successfully
Using device: cuda
PyTorch version: 2.6.0+gitdbfe118
GPU: AMD Radeon Graphics
GPU Memory: 68.7 GB


In [3]:
# 1. Tokenizer Setup - Using GPT-2 for Universal Access

print("=== Setting up Tokenizer ===")

# Use GPT-2 as it's widely available and demonstrates key concepts
model_name = 'gpt2'

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"Successfully loaded tokenizer: {model_name}")
    print(f"Vocabulary size: {len(tokenizer)}")
    print(f"Model max length: {tokenizer.model_max_length}")
    
    # Add padding token if it doesn't exist (GPT-2 doesn't have one by default)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print("Added padding token (using EOS token)")
        
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Please ensure you have internet connection or the model is cached locally")

=== Setting up Tokenizer ===
Successfully loaded tokenizer: gpt2
Vocabulary size: 50257
Model max length: 1024
Added padding token (using EOS token)


## 1. Basic Tokenization Concepts

Understanding how text is converted to numbers that neural networks can process.

In [4]:
# 1.1 Text to Token Conversion

print("=== Basic Tokenization Examples ===")

# Test sentences with different characteristics
test_texts = [
    "Hello, world!",                    # Simple English
    "The quick brown fox jumps.",       # Common English words
    "Machine learning is fascinating.", # Technical terms
    "OpenAI's GPT-2 model",            # Mixed case, apostrophes
    "2023年人工智能发展",                 # Mixed languages/characters
    "COVID-19 pandemic affected everyone.", # Numbers, acronyms
]

for i, text in enumerate(test_texts, 1):
    print(f"\nExample {i}: '{text}'")
    
    # Tokenize the text
    tokens = tokenizer.tokenize(text)
    print(f"  Tokens: {tokens}")
    print(f"  Number of tokens: {len(tokens)}")
    
    # Convert to IDs
    token_ids = tokenizer.encode(text)
    print(f"  Token IDs: {token_ids}")
    
    # Decode back to text
    decoded = tokenizer.decode(token_ids)
    print(f"  Decoded: '{decoded}'")
    print(f"  Perfect reconstruction: {text == decoded.strip()}")

print(f"\nKey Observations:")
print(f"- Punctuation often becomes separate tokens")
print(f"- Subword tokenization breaks unknown/rare words")
print(f"- Numbers and special characters need special handling")
print(f"- Different languages may tokenize differently")

=== Basic Tokenization Examples ===

Example 1: 'Hello, world!'
  Tokens: ['Hello', ',', 'Ġworld', '!']
  Number of tokens: 4
  Token IDs: [15496, 11, 995, 0]
  Decoded: 'Hello, world!'
  Perfect reconstruction: True

Example 2: 'The quick brown fox jumps.'
  Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', '.']
  Number of tokens: 6
  Token IDs: [464, 2068, 7586, 21831, 18045, 13]
  Decoded: 'The quick brown fox jumps.'
  Perfect reconstruction: True

Example 3: 'Machine learning is fascinating.'
  Tokens: ['Machine', 'Ġlearning', 'Ġis', 'Ġfascinating', '.']
  Number of tokens: 5
  Token IDs: [37573, 4673, 318, 13899, 13]
  Decoded: 'Machine learning is fascinating.'
  Perfect reconstruction: True

Example 4: 'OpenAI's GPT-2 model'
  Tokens: ['Open', 'AI', "'s", 'ĠG', 'PT', '-', '2', 'Ġmodel']
  Number of tokens: 8
  Token IDs: [11505, 20185, 338, 402, 11571, 12, 17, 2746]
  Decoded: 'OpenAI's GPT-2 model'
  Perfect reconstruction: True

Example 5: '2023年人工智能发展'
  Tokens: ['20', 

In [5]:
# 1.2 Multilingual Tokenization Analysis

print("=== Multilingual Text Processing ===")

# Test with different languages
multilingual_texts = {
    "English": "Shanghai is a famous city in China.",
    "Chinese": "上海是中国一座有名的城市。",
    "Mixed": "上海科技大学 (ShanghaiTech University) is located in China.",
    "Technical": "NLP (Natural Language Processing) 自然语言处理"
}

for lang, text in multilingual_texts.items():
    print(f"\n{lang} Text: '{text}'")
    
    # Detailed tokenization analysis
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text, return_tensors="pt")
    
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print(f"  Token IDs: {token_ids[0].tolist()}")
    
    # Analyze each token individually
    for i, (token, token_id) in enumerate(zip(tokens, tokenizer.encode(text))):
        decoded_single = tokenizer.decode([token_id])
        print(f"    [{i}] '{token}' -> ID {token_id} -> '{decoded_single}'")
    
    # Check decoding accuracy
    decoded_full = tokenizer.decode(token_ids[0])
    print(f"  Reconstructed: '{decoded_full}'")
    
    # Calculate efficiency
    char_count = len(text)
    token_count = len(tokens)
    compression_ratio = char_count / token_count if token_count > 0 else 0
    print(f"  Compression: {char_count} chars -> {token_count} tokens (ratio: {compression_ratio:.2f})")

print(f"\nTokenization Insights:")
print(f"- English words often map to single tokens")
print(f"- Chinese characters may need multiple tokens")
print(f"- Mixed language text requires careful handling")
print(f"- Compression ratio varies by language and content")

=== Multilingual Text Processing ===

English Text: 'Nanjing University is a prestigious institution.'
  Tokens (9): ['N', 'an', 'jing', 'ĠUniversity', 'Ġis', 'Ġa', 'Ġprestigious', 'Ġinstitution', '.']
  Token IDs: [45, 272, 49940, 2059, 318, 257, 23566, 9901, 13]
    [0] 'N' -> ID 45 -> 'N'
    [1] 'an' -> ID 272 -> 'an'
    [2] 'jing' -> ID 49940 -> 'jing'
    [3] 'ĠUniversity' -> ID 2059 -> ' University'
    [4] 'Ġis' -> ID 318 -> ' is'
    [5] 'Ġa' -> ID 257 -> ' a'
    [6] 'Ġprestigious' -> ID 23566 -> ' prestigious'
    [7] 'Ġinstitution' -> ID 9901 -> ' institution'
    [8] '.' -> ID 13 -> '.'
  Reconstructed: 'Nanjing University is a prestigious institution.'
  Compression: 48 chars -> 9 tokens (ratio: 5.33)

Chinese Text: '南京大学是一所著名的学府。'
  Tokens (22): ['åį', 'Ĺ', 'äº', '¬', 'å¤§', 'åŃ', '¦', 'æĺ¯', 'ä¸Ģ', 'æī', 'Ģ', 'è', 'ĳ', 'Ĺ', 'åĲ', 'į', 'çļĦ', 'åŃ', '¦', 'åº', 'ľ', 'ãĢĤ']
  Token IDs: [39355, 245, 12859, 105, 32014, 27764, 99, 42468, 31660, 33699, 222, 164, 239, 245, 289

### Understanding the � Replacement Character

### What You're Seeing

When decoding individual tokens, you might see:

```
'åį' -> ID 39355 -> '�'
```

The � character appears because **GPT-2 uses Byte-Level BPE**, which splits characters into bytes.


### Why This Happens

**Multi-byte characters need multiple tokens:**

- English "A" = 1 byte = usually 1 token ✓
- Chinese "大" = 3 bytes = typically 2-3 tokens

**Single token = incomplete character:**

- Decoding one token alone → � (incomplete byte sequence)
- Decoding all tokens together → "大" (complete character) ✓

### Key Takeaway

 **Always decode complete token sequences**, not individual tokens

 **This is expected behavior**, not an error

 **Chinese text uses ~3x more tokens** than English for the same meaning


In [6]:
# 1.3 Tensor Processing and Batch Operations
TODO = None
print("=== Working with Token Tensors ===")

# Sample texts for batch processing
sample_texts = [
    "Artificial intelligence is transforming the world.",
    "Machine learning models need large datasets.",
    "Deep learning uses neural networks with many layers.",
]

# --- Step 1: Batch Tokenization ---
# TODO: Configure the tokenizer call to pad, truncate, and return PyTorch tensors.
print("Running batch processing...")
batch_encoded = tokenizer(
    sample_texts, padding=TODO, truncation=TODO, return_tensors=TODO
)

batch_input_ids = batch_encoded["input_ids"]
batch_attention_mask = batch_encoded["attention_mask"]

print(f"Batch input IDs shape: {batch_input_ids.shape}")
print(f"Batch attention mask shape: {batch_attention_mask.shape}")


# --- Step 2: Move Tensors to Device ---
# TODO: Move the input_ids and attention_mask tensors to the designated device.
if torch.cuda.is_available():
    batch_input_ids = TODO
    batch_attention_mask = TODO
    print(f"Batch tensors successfully moved to device: {batch_input_ids.device}")
else:
    print("CUDA not available. Tensors remain on CPU.")


# --- Step 3: Analyze Padding ---
# TODO: Loop through the batch and calculate the number of real vs. padding tokens for each sample.
print("\nAnalyzing padding per sample:")
for i in range(len(sample_texts)):
    # Hint: The attention mask contains 1s for real tokens and 0s for padding.
    # You can use tensor operations like .sum() to count them.

    # Get the mask for the current sample
    sample_mask = batch_attention_mask[i]

    # Calculate counts
    actual_tokens_count = TODO
    total_tokens_count = len(sample_mask)
    padded_tokens_count = TODO

    print(
        f"  Sample {i+1}: {actual_tokens_count} real tokens, {padded_tokens_count} padding tokens"
    )
print(f"\nKey Concepts:")
print(f"- Padding enables batch processing of variable-length sequences")
print(f"- Attention masks tell the model which tokens are real vs padding")
print(f"- GPU processing accelerates tokenization for large batches")
print(f"- Truncation handles sequences longer than model limits")

=== Working with Token Tensors ===
Individual Processing:

Text 1: 'Artificial intelligence is transforming the world.'
  Input IDs shape: torch.Size([1, 8])
  Input IDs: [8001, 9542, 4430, 318, 25449, 262, 995, 13]
  Attention mask: [1, 1, 1, 1, 1, 1, 1, 1]
  Moved to device: cuda:0

Text 2: 'Machine learning models need large datasets.'
  Input IDs shape: torch.Size([1, 7])
  Input IDs: [37573, 4673, 4981, 761, 1588, 40522, 13]
  Attention mask: [1, 1, 1, 1, 1, 1, 1]
  Moved to device: cuda:0

Text 3: 'Deep learning uses neural networks with many layers.'
  Input IDs shape: torch.Size([1, 9])
  Input IDs: [29744, 4673, 3544, 17019, 7686, 351, 867, 11685, 13]
  Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1]
  Moved to device: cuda:0

Batch Processing:
  Batch input IDs shape: torch.Size([3, 9])
  Batch attention mask shape: torch.Size([3, 9])
  Batch on device: cuda:0
  Sample 1: 8 real tokens, 1 padding tokens
  Sample 2: 7 real tokens, 2 padding tokens
  Sample 3: 9 real tokens, 0 padd

In [7]:
# 2. Tokenization Methods Comparison

print("=== Different Tokenization Approaches ===")

test_text = "The preprocessing step tokenizes input text efficiently."

print(f"Input text: '{test_text}'")
print(f"Character count: {len(test_text)}")

# Method 1: Direct tokenization
print(f"\n1. Direct Tokenization:")
tokens = tokenizer.tokenize(test_text)
print(f"  Tokens: {tokens}")
print(f"  Token count: {len(tokens)}")

# Method 2: Encoding to IDs
print(f"\n2. Encoding to IDs:")
ids = tokenizer.encode(test_text)
print(f"  Token IDs: {ids}")
print(f"  ID count: {len(ids)}")

# Method 3: Full encoding with special tokens
print(f"\n3. Full Encoding (with special tokens):")
full_encoded = tokenizer.encode(test_text, add_special_tokens=True)
print(f"  With special tokens: {full_encoded}")
print(f"  Special token IDs: start={tokenizer.bos_token_id}, end={tokenizer.eos_token_id}")

# Method 4: Token to ID conversion
print(f"\n4. Token to ID Conversion:")
token_to_id_result = tokenizer.convert_tokens_to_ids(tokens)
print(f"  Converted IDs: {token_to_id_result}")
print(f"  Matches direct encoding: {token_to_id_result == tokenizer.encode(test_text, add_special_tokens=False)}")

# Method 5: ID to Token conversion
print(f"\n5. ID to Token Conversion:")
id_to_token_result = tokenizer.convert_ids_to_tokens(ids)
print(f"  Converted tokens: {id_to_token_result}")

# Analyze special tokens
print(f"\n6. Special Token Analysis:")
special_tokens = {
    'PAD': tokenizer.pad_token_id,
    'UNK': tokenizer.unk_token_id, 
    'BOS': tokenizer.bos_token_id,
    'EOS': tokenizer.eos_token_id,
}

for name, token_id in special_tokens.items():
    if token_id is not None:
        token_str = tokenizer.decode([token_id])
        print(f"  {name} token: ID {token_id} -> '{token_str}'")
    else:
        print(f"  {name} token: Not defined")

print(f"\nTokenization Workflow:")
print(f"  Text -> tokenize() -> Tokens -> convert_tokens_to_ids() -> IDs")
print(f"  Text -> encode() -> IDs (direct)")
print(f"  IDs -> decode() -> Text (reconstruction)")
print(f"  IDs -> convert_ids_to_tokens() -> Tokens -> detokenize() -> Text")

=== Different Tokenization Approaches ===
Input text: 'The preprocessing step tokenizes input text efficiently.'
Character count: 56

1. Direct Tokenization:
  Tokens: ['The', 'Ġpre', 'processing', 'Ġstep', 'Ġtoken', 'izes', 'Ġinput', 'Ġtext', 'Ġefficiently', '.']
  Token count: 10

2. Encoding to IDs:
  Token IDs: [464, 662, 36948, 2239, 11241, 4340, 5128, 2420, 18306, 13]
  ID count: 10

3. Full Encoding (with special tokens):
  With special tokens: [464, 662, 36948, 2239, 11241, 4340, 5128, 2420, 18306, 13]
  Special token IDs: start=50256, end=50256

4. Token to ID Conversion:
  Converted IDs: [464, 662, 36948, 2239, 11241, 4340, 5128, 2420, 18306, 13]
  Matches direct encoding: True

5. ID to Token Conversion:
  Converted tokens: ['The', 'Ġpre', 'processing', 'Ġstep', 'Ġtoken', 'izes', 'Ġinput', 'Ġtext', 'Ġefficiently', '.']

6. Special Token Analysis:
  PAD token: ID 50256 -> '<|endoftext|>'
  UNK token: ID 50256 -> '<|endoftext|>'
  BOS token: ID 50256 -> '<|endoftext|>'
  EOS t

In [8]:
# 2.1 Vocabulary Analysis and Token Statistics

print("=== Tokenizer Vocabulary Analysis ===")

# Vocabulary statistics
vocab_size = TODO
print(f"Total vocabulary size: {vocab_size:,} tokens")

# Sample vocabulary entries
print(f"\nSample vocabulary entries:")
sample_ids = [0, 1, 2, 100, 1000, vocab_size-1]
for token_id in sample_ids:
    if token_id < vocab_size:
        token = TODO
        print(f"  ID {token_id:>6}: '{token}'")

# Most common tokens (usually shorter, more frequent)
print(f"\nFirst 20 tokens (typically most common):")
first_tokens = TODO
for token_id, token in first_tokens:
    print(f"  {token_id:>2}: '{token}'")

# Token length analysis
print(f"\nToken length analysis:")
sample_size = min(1000, vocab_size)
token_lengths = []
for i in range(sample_size):
    token_text = TODO
    token_lengths.append(len(token_text))

if token_lengths:
    avg_length = sum(token_lengths) / len(token_lengths)
    max_length = max(token_lengths)
    min_length = min(token_lengths)
    
    print(f"  Sample size: {sample_size} tokens")
    print(f"  Average token length: {avg_length:.2f} characters")
    print(f"  Token length range: {min_length} - {max_length} characters")

# Special character handling
print(f"\nSpecial character analysis:")
special_chars = ['!', '@', '#', '$', '%', '&', '*', '(', ')', '-', '_', '=', '+']
for char in special_chars:
    tokens = TODO
    ids = TODO
    print(f"  '{char}' -> tokens: {tokens} -> IDs: {ids}")

# Number handling
print(f"\nNumber tokenization:")
numbers = ['0', '123', '2023', '3.14', '1,000,000']
for num in numbers:
    tokens = TODO
    print(f"  '{num}' -> {len(tokens)} tokens: {tokens}")

print(f"\nVocabulary Insights:")
print(f"- Smaller IDs often represent more frequent tokens")
print(f"- Special characters may become individual tokens")
print(f"- Numbers can be split into multiple tokens")
print(f"- Subword tokenization handles rare/unknown words")
print(f"- Vocabulary size affects model memory and computation")

=== Tokenizer Vocabulary Analysis ===
Total vocabulary size: 50,257 tokens

Sample vocabulary entries:
  ID      0: '!'
  ID      1: '"'
  ID      2: '#'
  ID    100: '�'
  ID   1000: 'ale'
  ID  50256: '<|endoftext|>'

First 20 tokens (typically most common):
   0: '!'
   1: '"'
   2: '#'
   3: '$'
   4: '%'
   5: '&'
   6: '''
   7: '('
   8: ')'
   9: '*'
  10: '+'
  11: ','
  12: '-'
  13: '.'
  14: '/'
  15: '0'
  16: '1'
  17: '2'
  18: '3'
  19: '4'

Token length analysis:
  Sample size: 1000 tokens
  Average token length: 2.73 characters
  Token length range: 1 - 8 characters

Special character analysis:
  '!' -> tokens: ['!'] -> IDs: [0]
  '@' -> tokens: ['@'] -> IDs: [31]
  '#' -> tokens: ['#'] -> IDs: [2]
  '$' -> tokens: ['$'] -> IDs: [3]
  '%' -> tokens: ['%'] -> IDs: [4]
  '&' -> tokens: ['&'] -> IDs: [5]
  '*' -> tokens: ['*'] -> IDs: [9]
  '(' -> tokens: ['('] -> IDs: [7]
  ')' -> tokens: [')'] -> IDs: [8]
  '-' -> tokens: ['-'] -> IDs: [12]
  '_' -> tokens: ['_'] -> ID

## 3. Advanced Tokenization Topics

Exploring advanced concepts like subword algorithms, out-of-vocabulary handling, and tokenization strategies for different domains.

In [9]:
# 3.1 Subword Tokenization Analysis

print("=== Subword Tokenization Behavior ===")

# Test with various word types
test_words = [
    # Common words (likely single tokens)
    "the", "and", "is", "to", "in",
    
    # Uncommon/rare words (likely split)
    "preprocessing", "tokenization", "subword", "transformer",
    
    # Technical terms
    "PyTorch", "GPU", "CUDA", "neural", "networks",
    
    # Made-up words
    "superdupertokenizer", "pseudotechnical", "multidimensional",
    
    # Words with affixes
    "running", "walked", "happiness", "beautiful", "quickly"
]

def analyze_tokenization(word_list, tokenizer):
    """
    Analyzes and categorizes a list of words based on their tokenization by GPT-2.

    Args:
        word_list (list): A list of strings to analyze.
        tokenizer: An initialized Hugging Face tokenizer.

    Returns:
        tuple: A tuple containing:
            - categorized_words (dict): A dictionary grouping words by their token count.
            - most_fragmented_word (dict): A dictionary with the most fragmented word and its token count.
    """
    categorized_words = {}
    most_fragmented_word = {"word": "", "count": 0}

    # --- Step 1 & 2: Iterate and Categorize ---
    for word in word_list:
        # TODO: Tokenize the current word to get its subword tokens.
        # Note: Do not add special tokens like [CLS] or [SEP].
        tokens = TODO

        # TODO: Get the number of tokens.
        num_tokens = TODO

        # TODO: Populate the categorized_words dictionary.
        # If the key (num_tokens) doesn't exist, create it with a new list.
        # Then, append the current word to the correct list.
        if num_tokens not in categorized_words:
            TODO
        TODO

        # --- Step 3: Find the Most Fragmented Word ---
        # TODO: Check if the current word has more tokens than the one stored
        # in most_fragmented_word. If so, update the dictionary.
        if num_tokens > most_fragmented_word["count"]:
            TODO
            TODO

    # --- Step 4: Return the completed data structures ---
    return categorized_words, most_fragmented_word

# --- Execution ---
# Call the function and store the results.
analysis_results, most_split_word = analyze_tokenization(test_words, tokenizer)

# --- Verification ---
# The following print statements will help you verify your results.
print("--- Word Categorization by Token Count ---")
for count, words in sorted(analysis_results.items()):
    print(f"{count} Tokens: {words}")

print("\n--- Most Fragmented Word ---")
print(f"Word: '{most_split_word['word']}' -> {most_split_word['count']} tokens")

print(f"\nSubword Algorithm Benefits:")
print(f"- Handles infinite vocabulary with finite token set")
print(f"- Breaks unknown words into recognizable parts")
print(f"- Balances vocabulary size vs token sequence length")
print(f"- Enables cross-lingual transfer (shared subwords)")
print(f"- Reduces out-of-vocabulary problems")

=== Subword Tokenization Behavior ===
Word tokenization analysis:
  'the' -> ['the'] (Single token (common/known))
  'and' -> ['and'] (Single token (common/known))
  'is' -> ['is'] (Single token (common/known))
  'to' -> ['to'] (Single token (common/known))
  'in' -> ['in'] (Single token (common/known))
  'preprocessing' -> ['pre', 'processing'] (Split into 2 subwords)
    Reconstructed: 'preprocessing' (matches: True)
  'tokenization' -> ['token', 'ization'] (Split into 2 subwords)
    Reconstructed: 'tokenization' (matches: True)
  'subword' -> ['sub', 'word'] (Split into 2 subwords)
    Reconstructed: 'subword' (matches: True)
  'transformer' -> ['trans', 'former'] (Split into 2 subwords)
    Reconstructed: 'transformer' (matches: True)
  'PyTorch' -> ['Py', 'Tor', 'ch'] (Split into 3 subwords)
    Reconstructed: 'PyTorch' (matches: True)
  'GPU' -> ['GPU'] (Single token (common/known))
  'CUDA' -> ['CU', 'DA'] (Split into 2 subwords)
    Reconstructed: 'CUDA' (matches: True)
  'neu

In [10]:
# 3.2 Tokenization Performance and Efficiency

print("=== Tokenization Performance Analysis ===")

import time

# Test with different text lengths
test_texts = [
    "Short text.",
    "This is a medium length sentence with several words to tokenize and process efficiently.",
    " ".join(["This is a much longer text that repeats the same content multiple times."] * 10),
    " ".join(["Very long text with repeated content for performance testing."] * 50)
]

print("Performance comparison:")
for i, text in enumerate(test_texts):
    char_count = len(text)
    
    # Time the tokenization
    start_time = time.time()
    tokens = tokenizer.tokenize(text)
    tokenize_time = time.time() - start_time
    
    # Time the encoding
    start_time = time.time()
    ids = tokenizer.encode(text)
    encode_time = time.time() - start_time
    
    # Time the decoding
    start_time = time.time()
    decoded = tokenizer.decode(ids)
    decode_time = time.time() - start_time
    
    print(f"\nText {i+1} ({char_count} chars, {len(tokens)} tokens):")
    print(f"  Tokenize: {tokenize_time*1000:.2f}ms")
    print(f"  Encode:   {encode_time*1000:.2f}ms") 
    print(f"  Decode:   {decode_time*1000:.2f}ms")
    print(f"  Chars/token ratio: {char_count/len(tokens):.2f}")

# Batch vs individual processing
print(f"\nBatch vs Individual Processing:")
batch_texts = ["Sample text number " + str(i) for i in range(100)]

# Individual processing
start_time = time.time()
individual_results = [tokenizer.encode(text) for text in batch_texts]
individual_time = time.time() - start_time

# Batch processing
start_time = time.time()
batch_result = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
batch_time = time.time() - start_time

print(f"  Individual processing: {individual_time*1000:.2f}ms for {len(batch_texts)} texts")
print(f"  Batch processing: {batch_time*1000:.2f}ms for {len(batch_texts)} texts")
print(f"  Speedup: {individual_time/batch_time:.2f}x faster with batching")

# Memory usage analysis
print(f"\nMemory Usage Analysis:")
sample_text = "This is a sample text for memory analysis. " * 20

# Create different representations
tokens = tokenizer.tokenize(sample_text)
ids = tokenizer.encode(sample_text)
tensor_ids = tokenizer(sample_text, return_tensors="pt")["input_ids"]

print(f"  Original text: {len(sample_text)} characters")
print(f"  Token list: {len(tokens)} tokens (Python list)")
print(f"  ID list: {len(ids)} IDs (Python list)")
print(f"  Tensor: {tensor_ids.shape} (PyTorch tensor)")

if torch.cuda.is_available():
    gpu_tensor = tensor_ids.to(device)
    print(f"  GPU tensor: {gpu_tensor.shape} on {gpu_tensor.device}")

print(f"\nEfficiency Insights:")
print(f"- Batch processing significantly improves throughput")
print(f"- GPU tensors enable parallel processing")
print(f"- Subword tokenization balances vocabulary size and sequence length")
print(f"- Caching tokenizer results can improve repeated processing")
print(f"- Memory usage scales with sequence length and batch size")

=== Tokenization Performance Analysis ===
Performance comparison:

Text 1 (11 chars, 3 tokens):
  Tokenize: 0.91ms
  Encode:   0.05ms
  Decode:   0.06ms
  Chars/token ratio: 3.67

Text 2 (88 chars, 16 tokens):
  Tokenize: 0.26ms
  Encode:   0.20ms
  Decode:   0.03ms
  Chars/token ratio: 5.50

Text 3 (729 chars, 140 tokens):
  Tokenize: 0.36ms
  Encode:   0.18ms
  Decode:   0.03ms
  Chars/token ratio: 5.21

Text 4 (3099 chars, 500 tokens):
  Tokenize: 0.64ms
  Encode:   0.56ms
  Decode:   0.10ms
  Chars/token ratio: 6.20

Batch vs Individual Processing:
  Individual processing: 1.71ms for 100 texts
  Batch processing: 1.14ms for 100 texts
  Speedup: 1.50x faster with batching

Memory Usage Analysis:
  Original text: 860 characters
  Token list: 181 tokens (Python list)
  ID list: 181 IDs (Python list)
  Tensor: torch.Size([1, 181]) (PyTorch tensor)
  GPU tensor: torch.Size([1, 181]) on cuda:0

Efficiency Insights:
- Batch processing significantly improves throughput
- GPU tensors enable

## 4. Tokenization for LLM Applications

Understanding how tokenization impacts Large Language Model training, inference, and performance.



**Your Task**:

1.  **Implement the `create_llm_prompt` function**: This function will programmatically construct a formatted prompt from three distinct parts: a system message, a user query, and a long context string.
2.  **Format the Prompt**: The final prompt must follow this exact structure, including the special tokens: `<|system|>{system_message}<|endoftext|><|user|>{context}\n\n{user_query}<|endoftext|><|assistant|>`
3.  **Tokenize and Analyze**: Tokenize the final combined prompt as well as each individual component (`system_message`, `context`, `user_query`).
4.  **Perform Context Window Analysis**: Calculate the total number of tokens in the final prompt. Determine if this total exceeds the `max_context_length`.
5.  **Calculate Component Contribution**: For each component, calculate what percentage of the *total* prompt's token count it occupies.
6.  **Return Structured Data**: The function must return a dictionary containing all the calculated metrics: the final prompt string, the token counts for each part and the total, a boolean indicating if the prompt is within the context limit, and the percentage contribution of each part.


In [11]:
# 4.1 LLM Tokenization Patterns and Best Practices

print("=== LLM Tokenization Patterns ===")


# Add special tokens that might be used in instruction-tuned models.
# This ensures they are treated as single tokens.
special_tokens_dict = {'additional_special_tokens': ['<|system|>', '<|user|>', '<|assistant|>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

# --- Data ---
# Define the components for our prompt.
system_message = "You are a helpful assistant that summarizes technical documents."
user_query = "Based on the provided text, what is the main innovation described?"
long_context = "Tokenization is the process of breaking down a stream of text into smaller units called tokens. These tokens can be words, subwords, or characters. For Large Language Models (LLMs), subword tokenization, like Byte-Pair Encoding (BPE), is common. It balances vocabulary size and the ability to handle unknown words. " * 8
max_window_size = 256 # A simulated small context window for testing.


def create_llm_prompt(system_message, user_query, context, tokenizer, max_context_length):
    """
    Builds and analyzes a formatted LLM prompt.

    Args:
        system_message (str): The system-level instruction for the model.
        user_query (str): The user's direct question.
        context (str): The context or document the query is about.
        tokenizer: An initialized Hugging Face tokenizer.
        max_context_length (int): The maximum number of tokens allowed.

    Returns:
        dict: A dictionary containing the full prompt and its analysis.
    """
    
    # --- Step 1 & 2: Format the Prompt ---
    # TODO: Construct the final prompt string using an f-string and the specified format.
    final_prompt = TODO

    # --- Step 3: Tokenize and Analyze ---
    # TODO: Tokenize the final prompt and each individual component.
    # Use add_special_tokens=False for components to get their raw token count.
    prompt_tokens = tokenizer.encode(final_prompt)
    system_tokens = tokenizer.encode(TODO, add_special_tokens=False)
    query_tokens = tokenizer.encode(TODO, add_special_tokens=False)
    context_tokens = tokenizer.encode(TODO, add_special_tokens=False)

    total_token_count = len(prompt_tokens)
    system_token_count = len(system_tokens)
    query_token_count = len(query_tokens)
    context_token_count = len(context_tokens)

    # --- Step 4: Context Window Analysis ---
    # TODO: Check if the total token count exceeds the max_context_length.
    is_within_limit = TODO

    # --- Step 5: Calculate Component Contribution ---
    # TODO: Calculate the percentage of the total tokens used by each part.
    # Handle the case where total_token_count might be zero.
    system_percentage = (system_token_count / total_token_count * 100) if total_token_count > 0 else 0
    query_percentage = TODO
    context_percentage = TODO
    
    # --- Step 6: Return Structured Data ---
    analysis = {
        "final_prompt": final_prompt,
        "total_tokens": total_token_count,
        "is_within_limit": is_within_limit,
        "component_analysis": {
            "system": {"tokens": system_token_count, "percentage": system_percentage},
            "user_query": {"tokens": query_token_count, "percentage": query_percentage},
            "context": {"tokens": context_token_count, "percentage": context_percentage},
        }
    }
    
    return analysis


# --- Execution ---
prompt_analysis = create_llm_prompt(system_message, user_query, long_context, tokenizer, max_window_size)

# --- Verification ---
print(f"--- Prompt Analysis (Max Window: {max_window_size}) ---")
print(f"Total Tokens: {prompt_analysis['total_tokens']}")
print(f"Fits in Context Window: {prompt_analysis['is_within_limit']}")
print("\n--- Component Breakdown ---")
for component, data in prompt_analysis['component_analysis'].items():
    print(f"  - {component.capitalize()}: {data['tokens']} tokens ({data['percentage']:.2f}%)")

print(f"\nLLM Tokenization Best Practices:")
print(f"- Monitor token counts to stay within model limits")
print(f"- Use appropriate special tokens for your model")
print(f"- Consider token efficiency when designing prompts")
print(f"- Batch process multiple inputs for efficiency")
print(f"- Handle different languages and domains appropriately")
print(f"- Account for tokenization differences between models")
print(f"- Use attention masks properly for padded sequences")
print(f"- Consider subword boundary effects on model understanding")

=== LLM Tokenization Patterns ===
LLM Input Pattern Analysis:

Example 1: Translate the following English text to French: 'H...
  Token count: 17
  Key tokens: ['Trans', 'late', 'Ġthe', 'Ġfollowing', 'ĠEnglish'] ... ['Ġyou', 'Ġtoday', "?'"]
  Words: 12, Token/Word ratio: 1.42

Example 2: Context: The capital of France is Paris. Question:...
  Token count: 18
  Key tokens: ['Context', ':', 'ĠThe', 'Ġcapital', 'Ġof'] ... ['Ġof', 'ĠFrance', '?']
  Words: 14, Token/Word ratio: 1.29

Example 3: Write a Python function that calculates the factor...
  Token count: 13
  Key tokens: ['Write', 'Ġa', 'ĠPython', 'Ġfunction', 'Ġthat'] ... ['Ġa', 'Ġnumber', ':']
  Words: 11, Token/Word ratio: 1.18

Example 4: Write a short story about a robot learning to pain...
  Token count: 11
  Key tokens: ['Write', 'Ġa', 'Ġshort', 'Ġstory', 'Ġabout'] ... ['Ġto', 'Ġpaint', ':']
  Words: 10, Token/Word ratio: 1.10

Example 5: Human: What is machine learning? Assistant: Machin...
  Token count: 12
  Key tokens: ['

## Summary and Key Takeaways

**What You've Learned:**

1. **Tokenization Fundamentals**: How text is converted to numerical representations
2. **Multiple Approaches**: Direct tokenization, encoding, decoding, and batch processing
3. **Multilingual Handling**: Challenges and solutions for different languages
4. **Vocabulary Management**: Understanding token IDs, special tokens, and vocabulary size
5. **Subword Algorithms**: How models handle unknown words and achieve vocabulary efficiency
6. **Performance Optimization**: Batch processing, GPU acceleration, and memory management
7. **LLM Applications**: Real-world patterns and best practices for language models

**Critical Concepts for LLM Development:**
- Tokenization is the bridge between human language and neural networks
- Subword tokenization enables handling of infinite vocabulary with finite tokens
- Proper padding and attention masks are essential for batch processing
- Token efficiency affects model performance and computational costs
- Different models may require different tokenization strategies

**Next Steps:**
- Experiment with different tokenizers (BERT, T5, LLaMA)
- Explore domain-specific tokenization challenges
- Learn about custom vocabulary creation and adaptation
- Understand tokenization impact on model fine-tuning
- Study multilingual and cross-lingual tokenization strategies